# Advancing and Optimizing RAG — Practice

> **Finish [`rag_pdf.ipynb`](./rag_pdf.ipynb) (the basic RAG practice) first.** The environment setup and kernel live there, and this reuses the same **RAG PDF Practice** kernel.

This is the practice for the blog post [Taking RAG Further](https://seojuny.dev/en/advanced-rag).

The approach is **cumulative**. Start from the simplest **basic RAG (STAGE 0)**, apply improvements **one at a time**, and **compare each answer with the previous stage**. See with your own eyes what each improvement fixes, then stack the next one on top. Once every stage is stacked, that is **advanced RAG**.

```
STAGE 0 basic → +table preservation → +structural chunking → +Contextual → +query rewriting → +hybrid → +reranking → +citation = advanced
```

Each stage is compared with **a question that exposes the effect of that improvement**. If the improvement changes the answer, we show the answer; if it only changes retrieval, we show the ranking and scores. Try changing the cell's `question` yourself — a recommended, effect-revealing question is noted in a comment.

The data is **7 internal-document PDFs of a fictional company, Novatech** (`../../data_advanced/en/`): employment rules, benefits, pay policy, security policy, HR policy, travel-expense policy, and an internal FAQ.

> Embedding and LLM calls carry a bit of randomness, so no two runs are 100% identical.

## Setup — LLM client
`get_client()`: OpenAI if `OPENAI_API_KEY` is set, otherwise local Ollama. (Returns the answer and embedding models together.)

In [ ]:
import os
import re
import math
from collections import Counter
from pathlib import Path

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()  # read OPENAI_API_KEY from the .env in the parent folder


def get_client():
    """OpenAI if OPENAI_API_KEY is set, otherwise local Ollama. (client, chat model, embedding model)"""
    if os.getenv("OPENAI_API_KEY", "").strip().startswith("sk-"):
        return OpenAI(), "gpt-4o-mini", "text-embedding-3-small"
    return OpenAI(base_url="http://localhost:11434/v1", api_key="ollama"), "qwen2.5", "nomic-embed-text"


client, CHAT_MODEL, EMBEDDING_MODEL = get_client()
print("answer:", CHAT_MODEL, "| embedding:", EMBEDDING_MODEL)

## Install libraries (once)

Install the libraries used in this practice **at compatible versions, in one go**. In particular, `langchain` differs greatly between the 1.x and 0.3 structures, so it is **pinned to 0.3** to mesh with the retriever components and `ragas`. (Skip if already installed.)

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "rank-bm25", "sentence-transformers",
    # langchain is pinned to the 0.3 line — 1.x requires langchain-core 1.x, which clashes with the retriever/ragas stack.
    # pinning langchain-core<1.0 resolves the whole ecosystem (langchain-openai, etc.) to 0.3-compatible versions.
    "langchain-core<1.0", "langchain<1.0", "langchain-community<0.4",
    "langchain-openai<1.0", "langchain-chroma<1.0", "langchain-text-splitters<1.0",
    "langchain-google-vertexai", "ragas==0.2.15"])
print("libraries ready")

## Shared functions — embed · index · search · answer · compare

The only things rebuilt at each stage are the 'index' (what is stored) and 'search' (how it is found). The embed, answer, and compare functions are defined once and reused.
Because it is an in-memory vector DB (`EphemeralClient`), the per-stage indexes stay isolated from one another (nothing is left on disk).

In [ ]:
import chromadb


def embed(texts):
    """Turn text (or a list of texts) into embedding vectors."""
    if isinstance(texts, str):
        texts = [texts]
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in response.data]


def make_index(items, name):
    """Embed items (=[{"text","source"}, ...]) into a searchable in-memory collection."""
    items = [it for it in items if it["text"].strip()]
    embeddings = embed([it["text"] for it in items])

    # EphemeralClient shares an in-memory backend, so if the same name exists, drop it and recreate.
    # (otherwise re-running the cell leaves old chunks behind and tangles retrieval)
    db = chromadb.EphemeralClient()
    try:
        db.delete_collection(name)
    except Exception:
        pass
    collection = db.get_or_create_collection(name, metadata={"hnsw:space": "cosine"})
    collection.add(
        ids=[f"chunk-{i}" for i in range(len(items))],
        embeddings=embeddings,
        documents=[it["text"] for it in items],
        metadatas=[{"source": it["source"]} for it in items],
    )
    return collection


def search(collection, query, k=3, where=None):
    """Find the k chunks closest in meaning to the query (use where to narrow to specific documents)."""
    result = collection.query(
        query_embeddings=[embed(query)[0]],
        n_results=k,
        where=where,
        include=["documents", "metadatas"],
    )
    documents = result["documents"][0]
    metadatas = result["metadatas"][0]
    return [{"text": text, "source": meta["source"]} for text, meta in zip(documents, metadatas)]


def answer_plain(question, chunks):
    """Answer plainly from the chunks (no citations)."""
    context = "\n\n".join(f"({c['source']}) {c['text']}" for c in chunks)
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": "Answer using only the internal documents; if it is not in the documents, say you don't know."},
            {"role": "user", "content": f"[Documents]\n{context}\n\n[Question]\n{question}"},
        ],
    )
    return response.choices[0].message.content


def compare_stage(question, before, after, before_label, after_label):
    """Print the 'before' and 'after' pipeline answers to the same question, side by side.
    before/after are functions that take a question and return an answer."""
    print("Q.", question)
    print(f"\n[{before_label}]\n" + before(question))
    print(f"\n[{after_label}]\n" + after(question))

## STAGE 0 — basic RAG (the starting point)

The same level of simple RAG as `rag_pdf.ipynb`: **plain pypdf extraction · character-count chunking · dense retrieval · plain answers.**
From here we stack improvements one by one. Ask for a value inside a table and basic RAG can't find it.

In [ ]:
import pypdf


def pypdf_text(path):
    """Join page text with pypdf (table structure breaks)."""
    pages = pypdf.PdfReader(path).pages
    return "\n".join(page.extract_text() or "" for page in pages)


def char_chunk(text, size=500, overlap=50):
    """Split simply by character count, overlapping by `overlap` to bridge boundary context."""
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return chunks


PDF_PATHS = sorted(Path("../../data_advanced/en").glob("*.pdf"))

# index v0: plain pypdf extraction + character-count chunking
items_v0 = [
    {"text": chunk, "source": pdf.stem}
    for pdf in PDF_PATHS
    for chunk in char_chunk(pypdf_text(str(pdf)))
]
col_v0 = make_index(items_v0, "stage0")


def rag_v0(question):
    """STAGE 0 basic RAG."""
    return answer_plain(question, search(col_v0, question, k=3))


# Change `question` and experiment.
question = "What is a General Manager's annual-salary equivalent?"
print(rag_v0(question))

## STAGE 1 — +table-preserving extraction

**Problem.** Basic RAG uses `pypdf`, which pulls text without distinguishing body from tables. A per-level table then runs together as `Staff KRW 3,000,000 Assistant Manager KRW 3,600,000 …` with no rows or columns, and the longer the table, the more the last rows get dropped from retrieval.

**Improvement.** A layout parser like `pymupdf4llm` turns tables into a **Markdown table with rows and columns intact**. You can see the effect directly in the extracted text.

> But the answer doesn't change yet — character-count chunking splits the table again. STAGE 2 fixes that.

In [ ]:
import pymupdf4llm

# Read each PDF as table-preserving Markdown (per-document source text)
RAW = {pdf.stem: pymupdf4llm.to_markdown(str(pdf), show_progress=False) for pdf in PDF_PATHS}

flat = pypdf_text("../../data_advanced/en/pay-policy.pdf")
markdown = RAW["pay-policy"]

print("─ pypdf (simple) ─ columns run together with no table structure:")
i = flat.find("Job Level")
print(flat[i:i + 95])

print("\n─ pymupdf4llm (layout) ─ rows and columns preserved as a Markdown table:")
j = markdown.find("|**Job Level**")
print(markdown[j:j + 150])

# index v1: pymupdf table-preserving extraction + (still) character-count chunking
items_v1 = [{"text": chunk, "source": source} for source, text in RAW.items() for chunk in char_chunk(text)]
col_v1 = make_index(items_v1, "stage1")


def rag_v1(question):
    """STAGE 1: table-preserving extraction + character-count chunking."""
    return answer_plain(question, search(col_v1, question, k=3))

## STAGE 2 — +structural chunking

**Problem.** Splitting by character count cuts the Markdown table apart again at a chunk boundary.

**Improvement.** Policy documents group content by `## heading` or by `Article N`. Cutting at those boundaries first keeps a table or clause whole in one chunk. Combined with table preservation (STAGE 1), a question about a value inside a table now gets answered.

Compare: previous v1 → this v2.

In [ ]:
def struct_chunk(text, max_size=700):
    """Cut at structural boundaries (## heading / Article N). If a piece is too long, fall back to character-count splitting."""
    chunks = []
    for part in re.split(r"(?=^#+ )|(?=Article \d+ \()", text, flags=re.M):
        part = part.strip()
        if not part:
            continue
        if len(part) <= max_size:
            chunks.append(part)
        else:
            chunks.extend(char_chunk(part, max_size, overlap=60))
    return chunks


# index v2: table-preserving extraction + structural chunking
items_v2 = [{"text": chunk, "source": source} for source, text in RAW.items() for chunk in struct_chunk(text)]
col_v2 = make_index(items_v2, "stage2")


def rag_v2(question):
    """STAGE 2: table preservation + structural chunking."""
    return answer_plain(question, search(col_v2, question, k=3))


# Change `question` and experiment. Recommended questions about a value inside a table:
#   - "How many added leave days does a Senior Manager get?"   (easy to confuse two adjacent tables)
#   - "What is a Manager's monthly base pay?"
question = "What is a General Manager's annual-salary equivalent?"
compare_stage(question, rag_v1, rag_v2, "v1 character-count chunking", "v2 +structural chunking")

## STAGE 3 — +Contextual Retrieval

**Problem.** Retrieval works at the chunk level, so a chunk torn from its document loses context. A chunk like "must be applied for **within 30 days** of the date the event occurs" doesn't actually contain the words 'family-event allowance' (those are only in the parent heading), so a vague question loses it in retrieval.

**Improvement.** [Contextual Retrieval](https://www.anthropic.com/news/contextual-retrieval) uses an LLM, before embedding, to write **one sentence about 'what in which document'** and prepend it to the chunk.

Compare: previous v2 → this v3.

In [ ]:
def add_context(chunk, source, full_doc):
    """Use an LLM to write one sentence about 'what in which document' and prepend it to the chunk (Contextual Retrieval)."""
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[{"role": "user", "content":
            f"<document name='{source}'>{full_doc}</document>\n"
            f"In one sentence, say what the chunk below is about within the document above.\nChunk:{chunk}"}],
    )
    context_line = response.choices[0].message.content.strip()
    return context_line + "\n" + chunk


# index v3: structural chunking + one context sentence per chunk (one LLM call per chunk at indexing time)
items_v3 = []
for source, full_doc in RAW.items():
    for chunk in struct_chunk(full_doc):
        if chunk.strip():
            items_v3.append({"text": add_context(chunk, source, full_doc), "source": source})
col_v3 = make_index(items_v3, "stage3")


# Pull a few examples to see how the Contextual sentence actually got attached.
print("─ Contextual: 'attached context sentence' + 'original chunk' examples ─")
shown = 0
for item in items_v3:
    context_line, _, original = item["text"].partition("\n")
    if len(original) < 120:            # skip very short chunks (the difference is hard to see)
        continue
    print("\n" + "=" * 70)
    print(f"[source] {item['source']}")
    print(f"[attached context] {context_line}")
    print(f"[original chunk] {original[:200].strip()} …")
    shown += 1
    if shown == 3:
        break
print("=" * 70)


def rag_v3(question):
    """STAGE 3: + Contextual."""
    return answer_plain(question, search(col_v3, question, k=3))


# Recommended vague questions whose chunk is missing the key term:
#   - "Within how many days of the event date must it be applied for?"
#   - "What period is the 90 days?"
question = "What has to be applied for within 30 days?"
compare_stage(question, rag_v2, rag_v3, "v2 no context", "v3 +Contextual")

## STAGE 4 — +query rewriting

From here the index (v3) stays fixed and we **polish the question before searching.**

**Problem.** An employee's question isn't always in a search-friendly form. It is often colloquial, or uses everyday words that differ from the document's formal terms — "working late" for the document's "overtime", "take home" for "removal". Embedded as-is, such a question can drift toward unrelated documents.

**Improvement.** The most basic query transformation is **query rewriting**. Ask the LLM to rewrite the question into a short query of the formal key nouns the document actually uses. "Do I get extra money for working late?" becomes "overtime allowance", which lands squarely on the **pay policy**.

Compare: previous v3 → this v4.

In [ ]:
def rewrite(question):
    """Rewrite the question into a short, search-friendly query."""
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[{"role": "user", "content":
            f"Rewrite the following question into a short search query of key nouns, good for searching internal policy documents. "
            f"Output only the query, no explanation.\nQuestion:{question}"}],
    )
    return response.choices[0].message.content.strip()


def retrieve_rewrite(question, k=4):
    """Rewrite the question, then search."""
    return search(col_v3, rewrite(question), k=k)


def rag_v4(question):
    """STAGE 4: + query rewriting."""
    return answer_plain(question, retrieve_rewrite(question))


# Change `question` and experiment. Recommended colloquial/mismatched questions (rewriting maps everyday words to the document's formal terms):
#   - "Can I take my laptop home?"      → "laptop removal policy"
#   - "When does my paycheck arrive?"   → "payday"
question = "Do I get extra money for working late?"
rewritten = rewrite(question)
v3_hits = search(col_v3, question, k=3)     # search with the original question
v4_hits = search(col_v3, rewritten, k=4)    # search with the rewritten question (used for both source and answer)

print("original question:", question)
print("rewritten:", rewritten)
print("\nv3 original-question sources:", [h["source"] for h in v3_hits])
print("v4 rewritten-question sources:", [h["source"] for h in v4_hits])
print("\n[v3 original question as-is]\n" + answer_plain(question, v3_hits))
print("\n[v4 +query rewriting]\n" + answer_plain(question, v4_hits))

### Bonus — query decomposition (implement it yourself)

If rewriting makes 'one question better', **decomposition** finds 'several pieces of information separately'. A question like "How much is A and how many days is B?" that must combine two or three documents leans to one side in a single search and misses the other. **Split** the question into per-information queries and search each, and you pull everything in.

The `decompose()` below is **a skeleton only**. Read the pseudocode comments, fill it in yourself, and compare with STAGE 4 rewriting.

In [ ]:
def decompose(question):
    """[Bonus] Split a compound question into 2-3 per-information queries (query decomposition).
    Fill in the pseudocode below and implement it yourself."""
    # 1. Ask the LLM to split the question into 2-3 key search queries (temperature=0)
    #    example prompt: "To answer the following question, what do you need to look up? Split into 2-3 short queries, one per line."
    # 2. Split the response by line (splitlines) and strip any leading number/symbol (1. - • etc.) from each line
    #    hint: re.sub(r"^[\s\d.)\-•]+", "", line).strip()
    # 3. Drop empty lines and return the list of queries
    raise NotImplementedError("implement decompose() yourself")


# Once implemented, uncomment retrieve_decompose below and compare with STAGE 4 rewriting:
# def retrieve_decompose(question):
#     hits, seen = [], set()
#     for sub_query in decompose(question):       # search separately for each split query
#         for hit in search(col_v3, sub_query, k=2):
#             if hit["text"] not in seen:
#                 seen.add(hit["text"]); hits.append(hit)
#     return hits
#
# Recommended question (information scattered across three documents — the effect is dramatic):
#   "Tell me an Assistant Manager's added leave days, the daily allowance for overseas Grade C, and a General Manager's welfare points"
#   → rewriting leans to one document with a single query; decomposition pulls all three: pay, travel, and benefits.
print("decompose() is still empty — read the pseudocode above and implement it yourself.")

## STAGE 5 — +hybrid search

**Problem.** Semantic (dense) search knows that "annual leave ≈ paid leave", but it often misses keywords that must match literally, like `SEC-VPN-01` or an employee ID or code. Conversely, keyword search (**BM25**) nails the exact characters but is weak on synonyms.

**Improvement.** Use both, and merge the rankings each produces with **RRF** (Reciprocal Rank Fusion). On this small, clean corpus the answer doesn't change, so we see the effect through the **retrieval ranking**.

Compare: previous v4 → this v5.

In [ ]:
# BM25 uses rank-bm25 (installed in the 'install libraries' cell above).
# Tokenization is simple lowercase word splitting — English needs no morphological analyzer.
from rank_bm25 import BM25Okapi


def tokenize(text):
    """Split into lowercase word tokens. Lowercasing lets codes like SEC-VPN-01 match too (→ 'sec','vpn','01')."""
    return re.findall(r"[a-z0-9]+", text.lower())


def reciprocal_rank_fusion(rankings, k=60):
    """Merge several search rankings into one score (RRF)."""
    scores = {}
    for ranking in rankings:
        for rank, index in enumerate(ranking):
            scores[index] = scores.get(index, 0) + 1 / (k + rank)  # higher rank (smaller number) → bigger score
    return sorted(scores, key=scores.get, reverse=True)


# Corpus for hybrid search — taken directly from col_v3.
# (always matches the text that search(col_v3) returns, so the text_to_index lookup never raises KeyError)
_stored = col_v3.get(include=["documents", "metadatas"])
adv_texts = _stored["documents"]
adv_sources = [m["source"] for m in _stored["metadatas"]]
text_to_index = {text: i for i, text in enumerate(adv_texts)}
bm25 = BM25Okapi([tokenize(text) for text in adv_texts])   # tokenize the chunks to build the BM25 index


def hybrid(query, k=3):
    """Combine the dense (semantic) ranking and the BM25 (keyword) ranking with RRF."""
    dense_ranking = [text_to_index[hit["text"]] for hit in search(col_v3, query, k=10)]

    bm25_scores = bm25.get_scores(tokenize(query))
    sparse_ranking = sorted(range(len(adv_texts)), key=lambda i: bm25_scores[i], reverse=True)[:10]

    fused = reciprocal_rank_fusion([dense_ranking, sparse_ranking])[:k]
    return [{"text": adv_texts[i], "source": adv_sources[i]} for i in fused]


def retrieve_hybrid(question, k=4):
    """Query rewriting + hybrid search."""
    return hybrid(rewrite(question), k=k)


def rag_v5(question):
    """STAGE 5: + hybrid."""
    return answer_plain(question, retrieve_hybrid(question))


# Change `code_query` and experiment. Recommended codes that must match exactly:
#   - "SEC-VPN-01" (remote VPN — dense already finds this one, so the orderings agree)
#   - "SEC-USB-##" (secure USB)
code_query = "SEC-NB-##"


def top3_labels(indices):
    """chunk-index list → '[source]#index' labels (the chunk index makes the ordering differences visible)."""
    return [f"{adv_sources[i]}#{i}" for i in indices[:3]]


dense_top = [text_to_index[hit["text"]] for hit in search(col_v3, code_query, k=3)]
bm25_scores = bm25.get_scores(tokenize(code_query))
bm25_top = sorted(range(len(adv_texts)), key=lambda i: bm25_scores[i], reverse=True)[:3]
hybrid_top = [text_to_index[hit["text"]] for hit in hybrid(code_query, k=3)]

print("dense  top3:", top3_labels(dense_top))
print("BM25   top3:", top3_labels(bm25_top))
print("hybrid top3:", top3_labels(hybrid_top))
print("\n→ dense and BM25 raise chunks in a different order, and hybrid merges the two with RRF.")
print("(the more real the document — lots of codes, little prose — the larger BM25's contribution)")

## STAGE 6 — +reranking

**Problem.** Vector search is fast but less precise, so the wider you retrieve, the more unrelated chunks slip in.

**Improvement.** Pull a generous set of candidates, then re-score them with a **cross-encoder** (`BAAI/bge-reranker-v2-m3`) that reads the question and chunk together as a pair, keeping only the top ones. The answer stays the same, but you see **how the candidates get reordered by score**.

Compare: previous v5 → this v6.

In [ ]:
# Reranking uses a real cross-encoder (sentence-transformers).
# (the first run downloads the BAAI/bge-reranker-v2-m3 model — a few hundred MB, once)
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")


def rerank(question, chunks, top_k=3):
    """Read (question, chunk) together with the cross-encoder to score relevance, and keep the top_k."""
    scores = reranker.predict([(question, c["text"]) for c in chunks])
    ranked = sorted(zip(scores, chunks), key=lambda pair: pair[0], reverse=True)
    return [chunk for _, chunk in ranked[:top_k]]


def retrieve_rerank(question, k=4):
    """Query rewriting + hybrid search to pull a generous candidate set, then narrow with reranking."""
    candidates = hybrid(rewrite(question), k=8)
    return rerank(question, candidates, top_k=k)


def rag_v6(question):
    """STAGE 6: + reranking."""
    return answer_plain(question, retrieve_rerank(question))


# Change `question` and experiment. Recommended questions where noise easily slips into the candidates:
#   - "Tell me the added leave days"
#   - "What are the password rules?"
# (here, to show the flow, we rerank the first-pass candidates directly. The real pipeline reranks the hybrid candidates.)
question = "What is the lodging cap on a business trip?"
candidates = search(col_v3, question, k=6)
print("─ 6 candidates before reranking (source) ─")
print("  ", [c["source"] for c in candidates])

print("\n─ top 3 after reranking (by cross-encoder score) ─")
scores = reranker.predict([(question, c["text"]) for c in candidates])
for score, chunk in sorted(zip(scores, candidates), key=lambda pair: pair[0], reverse=True)[:3]:
    snippet = chunk["text"].splitlines()[-1][:42].strip()
    print(f"   {score:.2f} [{chunk['source']}] {snippet} …")

## STAGE 7 — +citations (= advanced RAG complete)

**Problem.** Even when retrieval works, you may not know which document an answer came from, or the model may **hallucinate** content that isn't in the policy.

**Improvement.** Number each chunk, and make the model attach the evidence number `[n]` at the end of each sentence. With this, every STAGE is stacked — `rag_advanced` is now the **advanced RAG**.

Compare: previous v6 → this v7.

In [ ]:
def answer_cited(question, chunks):
    """Number the chunks and make the model attach the evidence number [n] at the end of each sentence."""
    context = "\n\n".join(f"[{i + 1}] ({c['source']}) {c['text']}" for i, c in enumerate(chunks))
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content":
                "Answer using only the internal documents; if it is not there, say you don't know. "
                "Put an evidence number like [1] at the end of each sentence."},
            {"role": "user", "content": f"[Documents]\n{context}\n\n[Question]\n{question}"},
        ],
    )
    return response.choices[0].message.content


def rag_advanced(question):
    """STAGE 7: + citations. The advanced RAG with every improvement stacked."""
    return answer_cited(question, retrieve_rerank(question))


# Change `question` and experiment. The answer is the same, but watch for the [n] evidence markers:
#   - "What is the lodging cap on a trip, by level?"
question = "How much is the allowance for my own wedding?"
compare_stage(question, rag_v6, rag_advanced, "v6 no citations", "v7 +citations")

### STAGE 0 → 7 at a glance — basic RAG vs advanced RAG

All eight stages are stacked. Take a colloquial question about a table value and feed it to both the **very first basic RAG** and the **finished advanced RAG** to see the difference.

In [ ]:
# Change `question` and experiment. Recommended questions basic gets stuck on but advanced solves (colloquial + a table value):
#   - "Can I take my laptop home?"
#   - "What is a General Manager's annual-salary equivalent?"
question = "If I'm a General Manager, what annual salary do I get?"
compare_stage(question, rag_v0, rag_advanced, "STAGE 0 basic RAG", "STAGE 7 advanced RAG")

# More to cover — agents, security, evaluation

On top of the cumulative stages, we add three things that are easy to skip in practice.

## Agentic RAG

**Problem.** So far "retrieve → generate" flowed one way, once. But a compound question scattered across different documents easily gets only one side answered from a single search.

**Improvement.** **Agentic RAG** doesn't finish retrieval in one shot. It looks at the results and judges for itself "is this enough, or should I look further?", repeating as needed (the search steps print to the screen).

In [ ]:
def agentic(question, max_steps=3):
    """Search with hybrid → judge 'is it enough?' → if not, change keywords and search again; if enough, answer."""
    gathered = []
    seen = set()
    sub_query = question

    for step in range(max_steps):
        for hit in hybrid(sub_query, k=3):
            if hit["text"] not in seen:
                seen.add(hit["text"])
                gathered.append(hit)

        context = "\n\n".join(f"({h['source']}) {h['text']}" for h in gathered)
        response = client.chat.completions.create(
            model=CHAT_MODEL,
            temperature=0,
            messages=[{"role": "user", "content":
                f"Question:{question}\nGathered documents:\n{context}\n\n"
                "Only when every part of the question is confirmed by the documents, output 'ANSWER: <answer>'. "
                "If some part still lacks evidence, output 'SEARCH: <keywords>' for that part only. Output one line."}],
        )
        line = response.choices[0].message.content.strip()

        if line.startswith("ANSWER:"):
            return line[len("ANSWER:"):].strip()

        sub_query = line.split("SEARCH:", 1)[-1].strip()
        print(f"  step{step} extra search: {sub_query}")

    return answer_cited(question, gathered)


# Change `question` and experiment. Recommended multi-hop questions that must cross different documents:
#   - "What is the lodging cap on a trip, and how often do I change my password?"
question = "How much is the childbirth congratulatory bonus, and how many days is maternity leave?"
print("[basic RAG]\n" + rag_v0(question))
print("\n[agentic RAG]")
print(agentic(question))

## Security and permissions

For **sensitive internal documents** like employment rules or pay, when a regular employee asks "what's someone else's salary?", the pay document must not surface in retrieval. Tag chunks with a permission at indexing time, and at search time **filter by the user's permission via metadata** so only viewable documents become candidates.

> Guarding against prompt injection: treat retrieved content as reference material only, never as instructions.

In [ ]:
# Per-document view permission (e.g. the pay policy requires the 'hr' role to view)
DOC_ACL = {"pay-policy": "hr"}


def answer_with_acl(question, user_roles, k=3):
    """Filter documents by the user's permission, and if it can't be answered for lack of permission, say so."""
    # 1) if a permission-gated document is the most relevant but the user lacks the permission → blocked notice
    top = search(col_v3, question, k=1)[0]
    needed = DOC_ACL.get(top["source"])
    if needed and needed not in user_roles:
        return f"'{top['source']}' requires view permission ({needed}) — you don't have it, so I can't answer."
    # 2) search only over viewable documents as candidates, then answer
    blocked = [src for src, role in DOC_ACL.items() if role not in user_roles]
    where = {"source": {"$nin": blocked}} if blocked else None
    return answer_cited(question, search(col_v3, question, k=k, where=where))


question = "What is a Manager's monthly base pay?"

print("─ regular employee (no permission) ─")
print("  answer:", answer_with_acl(question, user_roles=[]))

print("\n─ HR team (has hr permission) ─")
print("  answer:", answer_with_acl(question, user_roles=["hr"]))

## Evaluation — put a number on basic vs advanced

"You only know it really improved by measuring." The standard, without gold labels, is **LLM-as-Judge** — letting a strong LLM grade (in practice: RAGAS, DeepEval, TruLens).
For the same question, compare STAGE 0 retrieval and advanced retrieval on faithfulness and relevance.

In [ ]:
def judge(question, answer, chunks):
    """Grade whether the answer is grounded in the documents (faithfulness) and answers the question (relevance), each 0-1."""
    context = "\n\n".join(c["text"] for c in chunks)
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[{"role": "user", "content":
            f"Question:{question}\nDocuments:{context}\nAnswer:{answer}\n\n"
            "Grade faithfulness (grounded in the documents) and relevance (answers the question), each 0-1. Format: faithfulness=<num>, relevance=<num>"}],
    )
    return response.choices[0].message.content.strip()


question = "What is a General Manager's annual-salary equivalent?"
basic_hits = search(col_v0, question, k=3)
advanced_hits = retrieve_rerank(question)
basic_answer = answer_plain(question, basic_hits)
advanced_answer = answer_cited(question, advanced_hits)

print("question:", question)
print("\n[STAGE 0 basic] answer:", basic_answer)
print("  eval:", judge(question, basic_answer, basic_hits))
print("\n[advanced] answer:", advanced_answer)
print("  eval:", judge(question, advanced_answer, advanced_hits))

**When a new model comes out?** Even if a cheaper, better embedding appears, there's no guarantee it's better on *our* documents —
build this evaluation set once and, on every change, compare scores to adopt or roll back (a software **regression test**).

> **Building an evaluation set:** pull representative questions from real employee query logs, or have an LLM generate "question–evidence" pairs from each document and have a human review them. Mix in tricky types like tables and multi-document questions.

### Evaluate with RAGAS too (the practical standard)

The `judge` above is a minimal implementation to show the concept. In practice, a dedicated library like **[RAGAS](https://docs.ragas.io/)** grades faithfulness, relevance, and context precision/recall in a standardized way. Let's evaluate the same answer with RAGAS.

> ragas meshes with older langchain and clashes with the latest version, so we install a compatible version (`langchain<1.0`) alongside — the cell below sets it up automatically on first run.

In [ ]:
# Evaluate if RAGAS is installed (optional). If not, just print a note and skip.
try:
    from ragas import evaluate, EvaluationDataset
    from ragas.dataset_schema import SingleTurnSample
    from ragas.metrics import Faithfulness, ResponseRelevancy
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    RAGAS_AVAILABLE = True
except ImportError as e:
    RAGAS_AVAILABLE = False
    print("⚠️ RAGAS not installed — skipping:", e)
    print("   install: pip install 'ragas==0.2.15' 'langchain<1.0' 'langchain-community<0.4' langchain-google-vertexai langchain-openai")

if RAGAS_AVAILABLE:
    # Change `question` and evaluate.
    question = "How many added leave days does an Assistant Manager get, and what is the daily allowance for overseas Grade C?"
    chunks = retrieve_rerank(question)                  # the advanced RAG's retrieval result
    sample = SingleTurnSample(
        user_input=question,
        response=answer_cited(question, chunks),
        retrieved_contexts=[c["text"] for c in chunks],
    )
    judge_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
    judge_emb = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

    result = evaluate(
        EvaluationDataset(samples=[sample]),
        metrics=[Faithfulness(llm=judge_llm), ResponseRelevancy(llm=judge_llm, embeddings=judge_emb)],
    )
    scores = result.to_pandas()
    print("question :", question)
    print("answer   :", sample.response.replace("\n", " "))
    print(f"\n  faithfulness     : {scores['faithfulness'][0]:.3f}   (is the answer in the evidence documents)")
    print(f"  answer_relevancy : {scores['answer_relevancy'][0]:.3f}   (does the answer address the question)")

## Recap — what changed at each stage

| STAGE | Improvement applied | What got better | Cost |
|---|---|---|---|
| 1 | table-preserving extraction | table rows/cols survive (data) | layout-parser call |
| 2 | structural chunking | table/clause in one chunk → answers table questions | almost none |
| 3 | Contextual | even vague questions match the right chunk | one LLM call per chunk |
| 4 | query rewriting | colloquial/mismatched questions reach the right document | one LLM call per question |
| 5 | hybrid | codes and keywords handled reliably | managing BM25/tokenization |
| 6 | reranking | removes candidate noise, precision↑ | slow candidate re-scoring |
| 7 | citations | evidence check, hallucination↓ | almost none |

You don't need all of them. **Find the bottleneck by evaluation and add from that stage.**
For the concepts, see the blog post [Taking RAG Further](https://seojuny.dev/en/advanced-rag).

# Wrap-up — with a framework (LangChain) it's this short

Up to here we built the RAG pipeline function by function, by hand. In practice, a framework like **[LangChain](https://www.langchain.com/)** can do the same pipeline in a few lines. Loading, chunking, embedding, retrieval, and generation are already provided as components — you just wire them together.

So why not use a framework from the start? Because a framework hides the internals to build fast, but **if you don't know what's happening inside, you can't debug or improve it.** When retrieval pulls something irrelevant, is it the chunking or the embedding? Why isn't the table caught? Where should hybrid or reranking go? — you can pin these down only if you know the principles. What we took apart at each stage *is* those principles.

Below is the **entire advanced RAG** we stacked stage by stage, assembled identically from LangChain components. What we implemented by hand is built into the framework as parts — structural chunking (`RecursiveCharacterTextSplitter`), hybrid (`EnsembleRetriever`+`BM25Retriever`), query transformation (`MultiQueryRetriever`), reranking (`CrossEncoderReranker`), and citations. **Run it yourself** (auto-installs on first run). Which part does what, and where, you now know — because you built it by hand above.

In [ ]:
# Assemble the advanced RAG with LangChain (optional). Runs if the libraries are present, otherwise skips.
# (langchain is pinned to 0.3 — set in the 'install libraries' cell above.)

# A question to run yourself — feel free to change it.
question = "What is a General Manager's annual-salary equivalent, and how much is the allowance for my own wedding?"

try:
    from langchain.retrievers import EnsembleRetriever, ContextualCompressionRetriever
    from langchain.retrievers.document_compressors import CrossEncoderReranker
    from langchain.retrievers.multi_query import MultiQueryRetriever
    from langchain_community.document_loaders import PyMuPDFLoader
    from langchain_community.retrievers import BM25Retriever
    from langchain_community.cross_encoders import HuggingFaceCrossEncoder
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_openai import OpenAIEmbeddings, ChatOpenAI
    from langchain_chroma import Chroma
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.runnables import RunnablePassthrough
    from langchain_core.output_parsers import StrOutputParser
    LANGCHAIN_AVAILABLE = True
except ImportError as e:
    LANGCHAIN_AVAILABLE = False
    print("⚠️ LangChain not installed — skipping:", e)
    print("   install: pip install 'langchain-core<1.0' 'langchain<1.0' 'langchain-community<0.4' 'langchain-openai<1.0' 'langchain-chroma<1.0' 'langchain-text-splitters<1.0' rank-bm25 sentence-transformers")

if LANGCHAIN_AVAILABLE:
    from pathlib import Path
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    # STAGE 1 table preservation: load with PyMuPDFLoader (to preserve tables further, convert to Markdown with pymupdf4llm)
    docs = []
    for pdf in Path("../../data_advanced/en").glob("*.pdf"):
        docs += PyMuPDFLoader(str(pdf)).load()

    # STAGE 2 structural chunking: cut at Article N / ## heading boundaries first
    chunks = RecursiveCharacterTextSplitter(
        separators=["\n## ", "\nArticle ", "\n\n", "\n", " "], chunk_size=700, chunk_overlap=60
    ).split_documents(docs)

    store = Chroma.from_documents(chunks, OpenAIEmbeddings(model="text-embedding-3-small"))
    dense = store.as_retriever(search_kwargs={"k": 10})

    # STAGE 5 hybrid: bind dense + BM25 (reusing the STAGE 5 tokenizer above) with RRF
    bm25 = BM25Retriever.from_documents(chunks, preprocess_func=tokenize)
    bm25.k = 10
    hybrid_retriever = EnsembleRetriever(retrievers=[dense, bm25], weights=[0.5, 0.5])

    # STAGE 4 query transformation: make several versions of the question and search each (multi-query)
    multi_query = MultiQueryRetriever.from_llm(retriever=hybrid_retriever, llm=llm)

    # STAGE 6 reranking: re-score candidates with the cross-encoder (bge) and keep only the top 4
    reranker = CrossEncoderReranker(
        model=HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3"), top_n=4
    )
    retriever = ContextualCompressionRetriever(base_compressor=reranker, base_retriever=multi_query)

    # STAGE 7 citations: make it attach an evidence number at the end of each sentence
    def format_docs(docs):
        return "\n\n".join(f"[{i+1}] ({d.metadata.get('source', '')}) {d.page_content}" for i, d in enumerate(docs))

    prompt = ChatPromptTemplate.from_template(
        "Answer using only the internal documents; if it is not there, say you don't know. Put an evidence number like [1] at the end of each sentence."
        "\n\n[Documents]\n{context}\n\n[Question]\n{question}"
    )
    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    print(chain.invoke(question))

    # STAGE 3 Contextual has no stock component, so add it as a custom step that prepends a
    # context sentence to each chunk at indexing time (same idea as add_context above).

A framework is a tool you reach for **after** understanding the principles, for productivity. Reverse the order and, when something breaks, you're helpless.